<a href="https://www.kaggle.com/code/mrafraim/dl-day-48-fastapi-basics?scriptVersionId=314727244" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 48: FastAPI Basics
*Serving Models via API: Routes • Requests • Responses • /docs Testing*

Welcome to Day 48!

What You’ll Learn Today

1. What an API actually is (not just definition)
2. FastAPI fundamentals (routes, endpoints)
3. Request → processing → response cycle
4. How to expose a prediction function as an API
5. Using interactive /docs UI for testing

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# API

## What an API REALLY is

API = **a controlled interface that exposes your model as a service**

Not just “communication”.

It enforces:
- input format
- validation rules
- execution flow
- output contract


## Core Shift

### Before (Notebook Mindset)

You → call model → get output


### After (Production Mindset)

External system → API → model → response

👉 You are no longer the user<br>
👉 You are building the system others depend on

## End-to-End Flow

User uploads image<br>
↓<br>
API endpoint receives request<br>
↓<br>
Validate input (type, size, format)<br>
↓<br>
Preprocess (resize, normalize, tensor)<br>
↓<br>
Model inference<br>
↓<br>
Postprocess (class → label, add confidence)<br>
↓<br>
Return structured response


## What the API is responsible for

### 1. Input Contract

Example:

```json
{
  "file": "image.jpg"
}
```

If input is wrong:

👉 API must reject it (not crash)


### 2. Preprocessing Enforcement

* Same transforms as training
* Correct shape & dtype

👉 Prevents training-serving skew


### 3. Model Execution

* Calls inference pipeline
* Uses `model.eval()` + `no_grad()`

### 4. Output Contract

Example:

```json
{
  "class": "shirt",
  "confidence": 0.93
}
```

👉 Always structured, predictable


## Common Beginner Mistake

An API is NOT about the model.

It is about:

> making the model safely usable by other systems

They build:

API → directly call model → return raw output

Problems:

* no validation
* no preprocessing
* raw logits returned
* fragile system

API is NOT just “a way to call your model”.

It is:

> the production boundary that turns your model into a usable product

# Install & Setup FastAPI

## Why FastAPI?

- High performance (async by design)
- Minimal boilerplate
- Automatic API docs at `/docs`
- Well-suited for ML inference services


## Install Dependencies

```bash
pip install fastapi uvicorn
````

## Create Minimal API (main.py)

Create your first API endpoint.

This is NOT ML yet.
Just understand request/response.

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "API is running"}
```

## Run the Server

```bash
uvicorn main:app --reload
```
- main → file name
- app → FastAPI object
- --reload → auto restart on changes

## Access API

* Base URL: [http://127.0.0.1:8000](http://127.0.0.1:8000)

You should see:

{"message": "API is running"}

* Interactive Docs: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)

## What Just Happened

- FastAPI() → creates server app
- @app.get("/") → defines route
- function → handles request
- return → response (JSON)
  
## Common Mistakes

❌ Forgetting `--reload` → no auto-restart

❌ Running inside notebook cell → won’t behave correctly

❌ Port already in use → server fails silently


---
### <p style="text-align:center; color:red; font-size:18px;"> Break Down  (Optional) </p>

### 1. `app = FastAPI()`

```python
app = FastAPI()
```
> This creates your application

It defines:

* routes (`/predict`, `/`)
* request handling logic
* how inputs → outputs

But it does NOT run anything

Think:

> Blueprint of your API


### 2. `uvicorn app:app`

```bash
uvicorn main:app --reload
```
>This runs your application

It:

* starts a server
* opens a port (e.g., 8000)
* listens for requests
* sends them to your FastAPI app

Think:

> Engine that executes your API

### How they work together

```text
User Request
    ↓
Uvicorn (server)
    ↓
FastAPI app (your logic)
    ↓
Response
```

### Break down `uvicorn main:app`

```text
main:app
│   │
│   └── FastAPI object (app = FastAPI())
└────── file name (main.py)
```

Meaning:

> Go to `main.py`, find `app`, and run it”

### Critical Insight

| Component   | Role                       |
| ----------- | -------------------------- |
| `FastAPI()` | Defines WHAT should happen |
| `Uvicorn`   | Handles HOW it runs        |

---

# Request → Response Cycle

## What Really Happens

```text
Client (Browser / App)  
↓  
HTTP Request (GET / POST)  
↓  
Server (Uvicorn receives request)  
↓  
FastAPI matches route  
↓  
Python function executes  
↓  
Response converted to JSON  
↓  
Returned to client  
```

## Remember

- Every **endpoint** = a route (`/hello`)  
- Every **route** = a Python function  
- Every **function** = must return a response  

> No return = broken API


# Query Parameters

APIs are useless without input.

Query parameters let users send simple data via URL.

**Example**

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def say_hello(name: str):
    return {"message": f"Hello {name}"}
```

**How to Call It**

```text
http://127.0.0.1:8000/hello?name=Rahman
```

**What Happens Internally**

```text
Request URL:
GET /hello?name=Rahman
```

FastAPI does:

* extracts `name=Rahman`
* maps it to function argument `name`
* calls:

```python
say_hello(name="Rahman")
```

**Output**

```json
{
  "message": "Hello Rahman"
}
```

## Hidden Rules

### 1. Type Enforcement

```python
def say_hello(name: str)
```

> FastAPI validates type automatically

If wrong type:
→ returns error (not crash)

### 2. Missing Parameter

If user calls:

```text
/hello
```

Error:

```json
{
  "detail": "field required"
}
```

### 3. Multiple Parameters

```python
@app.get("/user")
def get_user(name: str, age: int):
    return {"name": name, "age": age}
```

Call:

```text
/user?name=Rahman&age=25
```

## Common Beginner Mistake

Thinking:

> “This is just string passing”

Wrong.

> This is API contract enforcement

* defines what input is allowed
* validates automatically
* rejects invalid requests

## Where This Goes Next

Query params are good for:

* small inputs (text, numbers)

BUT for ML:

- You won’t send images like this

You’ll use:

* POST requests
* file uploads

# POST Request

## Why POST?

| Method | Use Case |
|-------|--------|
| GET | Simple queries (small inputs via URL) |
| POST | Structured data (JSON, images, files) |

ML systems use **POST** because:
- inputs are large (images, text)
- inputs are structured
- URLs have size limits

## Core Concept

POST:  
> Send structured input → validate → process → return prediction


## Define Input Schema 

```python
from pydantic import BaseModel

class InputData(BaseModel):
    number: float
````

**What this REALLY does**

* Defines input contract
* Enforces type validation
* Rejects invalid requests automatically

👉 This is not optional in real APIs

## API Endpoint

```python
@app.post("/square")
def square(data: InputData):
    return {"result": data.number ** 2}
```

## Internal Flow (What FastAPI Does)

Client sends:

```json
{
  "number": 5
}
```

### Step-by-step:

1. Request hits `/square`
2. FastAPI reads JSON body
3. Converts it into `InputData`
4. Validates:

   * is `number` present?
   * is it a float?
5. Calls function:

```python
square(data=InputData(number=5))
```

6. Returns response


### Output

```json
{
  "result": 25
}
```


## Hidden Power

### Without schema (bad design)

```python
def square(data):
    return data["number"] ** 2
```

Problems:

* no validation
* crashes easily
* unclear input format


### With BaseModel

* strict input validation
* automatic error handling
* self-documented API (`/docs`)


## Error Handling Example

If user sends:

```json
{
  "number": "abc"
}
```

FastAPI returns:

```json
{
  "detail": [
    {
      "msg": "value is not a valid float"
    }
  ]
}
```

👉 No crash. Controlled failure.


##  Why This is CRITICAL for ML APIs

Because your real inputs will be:

* images (base64 / file upload)
* text (NLP)
* multiple features (tabular data)

Example:

```python
class PredictionInput(BaseModel):
    age: int
    income: float
    score: float
```

👉 This becomes your model input interface

## Mental Model Upgrade

```text
POST Request
   ↓
Schema Validation (BaseModel)
   ↓
Clean Python Object
   ↓
Model Inference
   ↓
Structured Response
```

## Strategic Insight

Your ML model is NOT your API.

👉 Your schema is the real interface

If schema is weak:

* system breaks
* users misuse API
* debugging becomes chaos


## Final Takeaway

POST + BaseModel =

> safe, structured, production-ready input pipeline for your model

# POST /predict (Image-Based ML API)

## Goal

User uploads image → API → model → prediction

## Step 1: Import required modules

```python
from fastapi import FastAPI, UploadFile, File
from PIL import Image
import torch
```

## Step 2: Create app

```python
app = FastAPI()
```

## Step 3: Define preprocessing (same as training)

```python
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
```

## Step 4: Load model (once)

```python
model = CNN()
model.load_state_dict(torch.load("model.pth", map_location="cpu"))
model.eval()
```

## Step 5: Create prediction endpoint

```python
@app.post("/predict")
def predict(file: UploadFile = File(...)):
    
    # 1. Read image
    image = Image.open(file.file).convert("L")
    
    # 2. Preprocess
    image = transform(image).unsqueeze(0)
    
    # 3. Inference
    with torch.no_grad():
        output = model(image)
        pred = output.argmax(dim=1)
    
    return {"prediction": int(pred.item())}
```

### What’s happening (simple, no confusion)

#### Input:

User uploads file

#### FastAPI gives you:

```python
file: UploadFile
```

👉 This is the image from user

#### You convert it:

```python
Image.open(file.file)
```

👉 Now it’s usable


#### Then:

```text
Image → transform → tensor → model → prediction
```

## Critical differences from our previous example

| Old (square) | New (ML)        |
| ------------ | --------------- |
| JSON input   | File input      |
| BaseModel    | UploadFile      |
| number       | image           |
| simple math  | model inference |

# /docs - Interactive API Interface (FastAPI)

When you run your API and open:

http://127.0.0.1:8000/docs

You get:
> an auto-generated interactive UI to test your API

Powered by:
- OpenAPI schema
- Swagger UI

**What’s Actually Happening**

FastAPI:
1. Reads your code (routes + schemas)
2. Builds a formal API specification
3. Renders it as an interactive interface

👉 This is NOT just UI<br>  
👉 It’s a contract visualization tool

## What You Can Do

### 1. Discover Endpoints
- See all routes (`GET`, `POST`)
- Understand required inputs

### 2. Send Requests (No frontend needed)

Click:
👉 “Try it out”

Then:
- enter input
- upload file (for ML APIs)
- execute request

### 3. Inspect Responses

You see:
- response JSON
- status code (200, 422, etc.)
- errors if any


## Why This Is Powerful (Real Reason)

### 1. Instant Validation Layer

If your API expects:

```python
class InputData(BaseModel):
    number: float
````

👉 `/docs` enforces it

Try wrong input → immediate error

### 2. Zero-Frontend Testing

Normally you’d need:

* frontend app
* Postman
* curl

👉 Here: none needed

### 3. Debugging Tool (Critical)

You can test:

* edge cases
* bad inputs
* large inputs

👉 Before any client integration

### 4. Contract Clarity

Anyone using your API can see:

* what to send
* what they’ll get

👉 Reduces integration errors massively

## Real ML System Flow

```text

User (mobile/web)
↓
API (FastAPI)
↓
Preprocessing
↓
Model (PyTorch)
↓
Prediction
↓
Response (JSON)
```

# Key Takeaways from Day 48
- FastAPI turns your model into a service
- Routes = entry points to your system
- Request → function → response is core loop
- /docs = fastest way to test APIs
- This is the bridge from ML → real-world usage

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
